In [1]:
import json
import os
import sys
import time
import xml.etree.ElementTree as ET

import requests
import yaml
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError

import openalex

In [17]:
def build_query():
    q = f"({CFG['query']})"
    journals = CFG.get("journals") or []
    if journals:
        q += " AND (" + " OR ".join(f'"{j}"[Journal]' for j in journals) + ")"
    return q


def search_pubmed():
    r = requests.get(f"{EUTILS}/esearch.fcgi", params={
        "db": "pubmed",
        "term": build_query(),
        "reldate": CFG["days_back"],
        "datetype": "edat",          # date the paper was added to PubMed
        "retmax": CFG.get("candidate_pool", 50),
        "sort": CFG.get("pubmed_sort", "relevance"),
        "retmode": "json",
        "api_key": NCBI_KEY,
    }, timeout=30)
    r.raise_for_status()
    return r.json()["esearchresult"]["idlist"]


def co_first_authors(authors):
    """[(position, last name)] of the authors PubMed marks as equal first authors.
    Only the run at the top of the list counts; flags further down usually mark co-senior authors."""
    run = []
    for i, a in enumerate(authors):
        if a.get("EqualContrib") != "Y":
            break
        run.append((i, a.findtext("LastName", "")))
    return run


def fetch_details(pmids):
    r = requests.get(f"{EUTILS}/efetch.fcgi", params={
        "db": "pubmed", "id": ",".join(pmids), "retmode": "xml",
        "api_key": NCBI_KEY,
    }, timeout=60)
    r.raise_for_status()
    root = ET.fromstring(r.content)

    papers = []
    for art in root.findall(".//PubmedArticle"):
        title_el = art.find(".//ArticleTitle")
        doi_el = art.find(".//PubmedData/ArticleIdList/ArticleId[@IdType='doi']")
        authors = art.findall(".//AuthorList/Author")
        papers.append({
            "pmid": art.findtext(".//PMID"),
            "doi": doi_el.text.strip().lower() if doi_el is not None and doi_el.text else None,
            "title": "".join(title_el.itertext()) if title_el is not None else "(no title)",
            "journal": art.findtext(".//Journal/Title") or "",
            "abstract": " ".join("".join(a.itertext())
                                 for a in art.findall(".//AbstractText")),
            "pubmed_authors": [f"{a.findtext('ForeName', '')} {a.findtext('LastName', '')}".strip()
                               for a in authors],
            "co_first": co_first_authors(authors),
                        "keywords": ["".join(k.itertext()).strip() for k in art.findall(".//KeywordList/Keyword")],
            "mesh": [d.text for d in art.findall(".//MeshHeading/DescriptorName")],
        })
    return papers


In [18]:
load_dotenv()
CFG = yaml.safe_load(open("config.yaml"))
RANK = CFG.get("ranking") or {}
DRY_RUN = "--dry-run" in sys.argv
SEEN_FILE = "seen.json"
EUTILS = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
NCBI_KEY = os.getenv("NCBI_API_KEY")   # optional; raises limit to 10 req/sec

kimi = OpenAI(base_url="https://api.moonshot.ai/v1",
              api_key=os.environ["MOONSHOT_API_KEY"])

In [19]:
a = build_query()
a

'(multi-omics AND diabetes)'

In [20]:
pmid = search_pubmed()[1]

In [21]:
fetch_details([pmid])

[{'pmid': '42675038',
  'doi': '10.1038/s41467-026-76153-8',
  'title': 'Genome-wide DNA methylation analysis revealed epigenetic mechanism underlying end-stage renal disease.',
  'journal': 'Nature communications',
  'abstract': 'End-stage renal disease (ESRD) remains a major clinical challenge with high morbidity and mortality, and its molecular mechanisms, particularly those shared among diverse primary kidney diseases during progression to ESRD, have not been studied. Here we conduct a large-scale two-stage epigenome-wide association study of ESRD in two independent cohorts consisting of 704 controls and 1031 ESRD cases. We identify 52 ESRD-associated differentially methylated CpG positions (ESRD DMPs) showing consistent association between the two cohorts and across diverse kidney diseases, implicating 144 candidate genes enriched in inflammatory and immune pathways. Five of the 52 DMPs are associated with ESRD complications, and seven with renal function decline in early-stage ch